# GFP 결합체 설계와 형광 변화 예측 · v0.1.6

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_protein/blob/main/GFP_Binder_T4.ipynb?v=0.1.6)

T4 GPU에서 전체 실행을 선택합니다. 기본 모드는 각 대조군의 GFP와 결합체 구조를 독립적으로 제공하고, recycle 10회와 시드 0·1·2로 결합 자세를 예측합니다. 체인 간 정답 거리 정보는 가립니다.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import subprocess
import sys

checkout = Path(mkdtemp(prefix="gfp_main_", dir="/content")) / "repo"
clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--single-branch", "--branch", "main",
     "https://github.com/HisameOgasahara/tmp_protein.git", str(checkout)],
    capture_output=True, text=True,
)
if clone.returncode:
    print(clone.stderr)
    clone.check_returncode()
revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
print(f"main: {revision[:12]}")

## 실행 모드

`BENCHMARK_ONLY`를 해제하면 뼈대 생성, 서열 설계, 복합체 예측, 형광 평가를 실행합니다. 전체 모드에서 자체 실험 데이터가 있으면 Colab 파일 패널에 CSV를 올리고 아래 경로를 입력합니다. CSV에는 `id,ratio,group,assay,source`와 `sequence` 또는 `pdb,entity`가 필요합니다. `ratio`는 동일 실험에서 측정한 기준 대비 형광 배율이고, `group`은 유사 결합체의 그룹입니다. 경로를 비워 두면 공개 대조군을 씁니다.

대조군 비교는 `target`·3회 → `target`·10회 → `both`·10회 순서로 실행합니다. `target`은 GFP 구조만, `both`는 두 단백질의 개별 구조를 제공합니다. 각 결과 ZIP을 보관하고 개선이 확인되면 시드를 `0,1,2,3,4,5,6,7,8,9`로 늘립니다.

In [ ]:
BENCHMARK_ONLY = True  # @param {type:"boolean"}
BENCHMARK_TEMPLATE = "both"  # @param ["target", "both"]
BENCHMARK_RECYCLES = 10  # @param [3, 10] {type:"raw"}
BENCHMARK_SEEDS = "0,1,2"  # @param {type:"string"}
ASSAY_CSV = ""  # @param {type:"string"}
command = [sys.executable, str(checkout / "gfp_pipeline/run.py"), "--revision", revision]
if BENCHMARK_ONLY:
    seeds = [int(value.strip()) for value in BENCHMARK_SEEDS.split(",")]
    command.extend(["--benchmark-only", "--benchmark-template", BENCHMARK_TEMPLATE,
                    "--benchmark-recycles", str(BENCHMARK_RECYCLES),
                    "--benchmark-seeds", *map(str, seeds)])
elif ASSAY_CSV.strip():
    command.extend(["--assays", str(Path(ASSAY_CSV).expanduser().resolve(strict=True))])
with subprocess.Popen(command, stdout=subprocess.PIPE, text=True) as process:
    for line in process.stdout:
        print(line.rstrip())
        if line.rstrip().endswith(".zip"):
            result_zip = Path(line.strip())
    if process.wait():
        raise RuntimeError(f"Pipeline failed with exit code {process.returncode}.")
result_dir = result_zip.with_suffix("")

## 실행 결과

기존 접촉면 기준 통과 여부와 ipTM 순위를 확인합니다. 결합체의 방향까지 판단하려면 GFP 정렬 후 결합체 RMSD(Å)와 Cα 잔기 쌍 접촉 회복률을 함께 봅니다. RMSD는 낮을수록, 회복률은 높을수록 기준 자세와 가깝습니다. 전체 모드의 형광 배율 1.0은 기준과 같은 형광입니다.

In [ ]:
import json
import pandas as pd

if (result_dir / "metrics/benchmark_summary.csv").is_file():
    config = json.loads((result_dir / "config/source.json").read_text())
    print(f"템플릿: {config['benchmark_template']}, recycles: {config['rf3']['n_recycles']}, seeds: {config['rf3_seeds']}")
    summary = pd.read_csv(result_dir / "metrics/benchmark_summary.csv")
    summary["status"] = summary["status"].map({
        "no_pose_recovered": "기준 충족 자세 없음",
        "pose_ranked_lower": "기준 충족 자세가 하위에 있음",
        "top_pose_recovered": "1위 자세가 기준 충족",
    })
    display(summary[["control", "status", "poses_recovered", "top_iptm_seed",
                     "top_iptm_contact_jaccard", "best_contact_jaccard",
                     "best_recovered_iptm_rank", "top_iptm_binder_rmsd_angstrom",
                     "top_iptm_native_ca_contact_pair_recovery"]].round(3))
    poses = pd.read_csv(result_dir / "metrics/benchmark_poses.csv")
    display(poses[["id", "seed", "iptm_rank", "iptm", "benchmark_contact_jaccard",
                   "target_ca_rmsd_angstrom",
                   "binder_ca_rmsd_after_target_alignment_angstrom",
                   "native_ca_contact_pair_recovery", "has_clash"]].round(3))
else:
    metrics = json.loads((result_dir / "metrics/fluorescence_model.json").read_text())
    status_text = {
        "ready": "형광 모델 검증 통과",
        "structural_benchmark_failed": "대조군 구조 검증 미통과",
        "insufficient_control_groups": "독립 실험 그룹 부족",
        "baseline_not_improved": "형광 변화 없음 기준 대비 개선 없음",
        "candidate_structure_failed": "후보 구조 검증 미통과",
        "outside_training_range": "접촉 특징이 학습 범위 밖",
        "outside_observed_effect_range": "예측값이 실험에서 관측한 범위 밖",
        "predicted": "예측값 산출",
    }
    print(status_text[metrics['validation']])
    if "held_out_mae_log2" in metrics:
        print(f"그룹별 검증 MAE (log2): {metrics['held_out_mae_log2']:.3f}")
        print(f"변화 없음 기준 MAE (log2): {metrics['no_effect_mae_log2']:.3f}")

    results = pd.read_csv(result_dir / "metrics/fluorescence_predictions.csv")
    results["prediction_status"] = results["prediction_status"].map(status_text)
    ratio = results["predicted_fluorescence_ratio"].round(3)
    results["predicted_fluorescence_ratio"] = ratio.astype(object).where(ratio.notna(), "보류")
    display(results[["id", "iptm", "prediction_status", "predicted_fluorescence_ratio"]])

In [ ]:
from google.colab import files

files.download(str(result_zip))